# Phase 4 - EDA + Visualization

Nine figures covering: time-series, weather/crime overlay, correlation heatmap, temp-bin distributions, monthly + day-of-week patterns, rain effect, per-category trends, and the COVID dip. Plus three statistical tests (t-test, ANOVA, Pearson with CI) and a per-category Pearson table.

In [1]:
import sys
sys.path.append("../src")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from viz import save_fig, temp_bin

sns.set_style("whitegrid")
sns.set_palette("colorblind")
plt.rcParams["figure.dpi"] = 120

df = pd.read_csv("../data/processed/merged.csv", parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)
df["month"] = df["date"].dt.month
df["day_of_week"] = df["date"].dt.dayofweek
df["temp_bin"] = df["tmax_f"].apply(temp_bin)
df["rain_status"] = np.where(df["prcp_mm"] > 1, "rainy", "dry")
print("loaded:", df.shape)

loaded: (2557, 22)


### Fig 1 - Daily crime + 30-day rolling average

In [2]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df["date"], df["total_crimes"], alpha=0.3, label="daily")
ax.plot(df["date"], df["total_crimes"].rolling(30).mean(), label="30-day rolling avg", linewidth=2)
ax.set_xlabel("date")
ax.set_ylabel("total crimes (count/day)")
ax.set_title("Chicago daily crime, 2018-2024")
ax.legend()
save_fig("fig_01_crime_timeseries")

### Fig 2 - TMAX overlaid with daily crime (dual y-axis)

In [3]:
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(df["date"], df["total_crimes"].rolling(14).mean(), color="tab:blue", label="crime (14-day avg)")
ax1.set_xlabel("date")
ax1.set_ylabel("total crimes (count/day)", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.plot(df["date"], df["tmax_f"].rolling(14).mean(), color="tab:red", alpha=0.7, label="TMAX (14-day avg)")
ax2.set_ylabel("TMAX (°F)", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")
ax2.grid(False)

plt.title("Daily crime vs. TMAX, both smoothed (14-day)")
save_fig("fig_02_temp_vs_crime")

### Fig 3 - Correlation heatmap (weather × crime categories)

In [4]:
wcols = ["tmax_f", "tmin_f", "prcp_mm", "snow_mm", "awnd_ms"]
ccols = ["total_crimes", "theft", "battery", "burglary", "assault", "narcotics", "criminal_damage"]
corr = df[wcols + ccols].corr().loc[wcols, ccols]

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("Pearson correlation: weather vs. crime")
save_fig("fig_03_corr_heatmap")

### Fig 4 - Crime distribution across temperature bins

In [5]:
order = ["cold", "cool", "mild", "warm", "hot"]
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x="temp_bin", y="total_crimes", order=order, ax=ax)
ax.set_xlabel("TMAX bin")
ax.set_ylabel("total crimes (count/day)")
ax.set_title("Daily crime by temperature bin")
save_fig("fig_04_temp_bins")

### Fig 5 - Average crime by month

In [6]:
monthly = df.groupby("month")["total_crimes"].mean()
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=monthly.index, y=monthly.values, ax=ax)
ax.set_xlabel("month")
ax.set_ylabel("avg crimes (count/day)")
ax.set_title("Average daily crime by month, 2018-2024")
save_fig("fig_05_avg_by_month")

### Fig 6 - Average crime by day of week

In [7]:
dow = df.groupby("day_of_week")["total_crimes"].mean()
labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=labels, y=dow.values, ax=ax)
ax.set_xlabel("day of week")
ax.set_ylabel("avg crimes (count/day)")
ax.set_title("Average daily crime by day of week")
save_fig("fig_06_avg_by_dow")

### Fig 7 - Rainy vs. dry day crime distributions

In [8]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.violinplot(data=df, x="rain_status", y="total_crimes", order=["dry", "rainy"], ax=ax)
ax.set_xlabel("rain status (rainy = > 1mm)")
ax.set_ylabel("total crimes (count/day)")
ax.set_title("Daily crime - rainy vs. dry days")
save_fig("fig_07_rain_violin")

### Fig 8 - Each crime category over time (30-day smoothed)

In [9]:
cats = ["theft", "battery", "burglary", "assault", "narcotics", "criminal_damage"]
fig, ax = plt.subplots(figsize=(12, 5))
for c in cats:
    ax.plot(df["date"], df[c].rolling(30).mean(), label=c)
ax.set_xlabel("date")
ax.set_ylabel("crimes (count/day, 30-day avg)")
ax.set_title("Crime categories over time, 2018-2024")
ax.legend()
save_fig("fig_08_categories_over_time")

### Fig 9 - COVID lockdown dip

Illinois issued the stay-at-home order on March 21, 2020. Plotting January-August 2020 to highlight the drop.

In [10]:
win = df[(df["date"] >= "2020-01-01") & (df["date"] <= "2020-08-31")]
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(win["date"], win["total_crimes"], alpha=0.5, label="daily")
ax.plot(win["date"], win["total_crimes"].rolling(7).mean(), label="7-day avg", linewidth=2)
ax.axvline(pd.to_datetime("2020-03-21"), color="red", linestyle="--", label="IL stay-at-home")
ax.annotate("lockdown begins", xy=(pd.to_datetime("2020-03-21"), 1100),
            xytext=(pd.to_datetime("2020-04-15"), 1200),
            arrowprops=dict(arrowstyle="->"))
ax.set_xlabel("date")
ax.set_ylabel("total crimes (count/day)")
ax.set_title("Crime around the COVID-19 stay-at-home order, 2020")
ax.legend()
save_fig("fig_09_covid_dip")

## Statistical tests

Three formal tests, all printed for the report.

In [11]:
# Welch's t-test: rainy vs dry total crime
rainy = df[df["rain_status"] == "rainy"]["total_crimes"]
dry = df[df["rain_status"] == "dry"]["total_crimes"]
t, p = stats.ttest_ind(rainy, dry, equal_var=False)
print(f"Welch t-test (rainy vs dry total crime):")
print(f"  rainy mean = {rainy.mean():.2f} (n={len(rainy)})")
print(f"  dry   mean = {dry.mean():.2f} (n={len(dry)})")
print(f"  t = {t:.3f}, p = {p:.4g}")

Welch t-test (rainy vs dry total crime):
  rainy mean = 658.37 (n=667)
  dry   mean = 675.48 (n=1890)
  t = -3.508, p = 0.000468


In [12]:
# one-way ANOVA across temp bins
groups = [df[df["temp_bin"] == b]["total_crimes"] for b in ["cold", "cool", "mild", "warm", "hot"]]
f, p = stats.f_oneway(*groups)
print("ANOVA (total crime across temp bins):")
for b, g in zip(["cold", "cool", "mild", "warm", "hot"], groups):
    print(f"  {b:5s} mean = {g.mean():.2f} (n={len(g)})")
print(f"  F = {f:.3f}, p = {p:.4g}")

ANOVA (total crime across temp bins):
  cold  mean = 568.60 (n=192)
  cool  mean = 633.36 (n=660)
  mild  mean = 657.16 (n=687)
  warm  mean = 720.91 (n=670)
  hot   mean = 730.26 (n=348)
  F = 157.258, p = 2.027e-120


In [13]:
# Pearson correlation TMAX vs total crime, with 95% CI via Fisher z
r, p = stats.pearsonr(df["tmax_f"], df["total_crimes"])
n = len(df)
z = np.arctanh(r)
se = 1 / np.sqrt(n - 3)
lo, hi = np.tanh(z - 1.96 * se), np.tanh(z + 1.96 * se)
print(f"Pearson r (TMAX vs total_crimes): {r:.3f}")
print(f"  95% CI: [{lo:.3f}, {hi:.3f}]")
print(f"  p = {p:.4g}, n = {n}")

Pearson r (TMAX vs total_crimes): 0.453
  95% CI: [0.422, 0.483]
  p = 1.191e-129, n = 2557


In [14]:
# per-category Pearson table
rows = []
for c in ["total_crimes", "theft", "battery", "burglary", "assault", "narcotics", "criminal_damage"]:
    r, p = stats.pearsonr(df["tmax_f"], df[c])
    rows.append({"category": c, "pearson_r": round(r, 3), "p_value": p})
pd.DataFrame(rows)

,category,pearson_r,p_value
0,total_crimes,0.453,1.190524e-129
1,theft,0.313,4.304745e-59
2,battery,0.493,1.297323e-156
3,burglary,0.104,1.280098e-07
4,assault,0.563,3.426466e-214
5,narcotics,-0.107,5.090314e-08
6,criminal_damage,0.501,8.240611e-163


## What I found

- **Strong seasonality.** Average daily crime climbs from ~611 in February to ~736 in July — about a 20% summer-vs-winter swing. The temp-bin ANOVA backs this up: cold-day mean is 569 vs 730 on hot days (F = 157.3, p ≈ 2e-120).
- **Temperature correlates positively with most categories, but the effect is uneven.** Pearson r with TMAX is 0.45 for total crime (95% CI [0.422, 0.483], p ≈ 1e-129). Assault leads at 0.56, followed by criminal damage (0.50), battery (0.49), and theft (0.31).
- **Burglary and narcotics aren't weather-driven.** Burglary r = 0.10, narcotics r = -0.11 (slightly *negative*). Whatever drives those categories isn't the kind of weather I have features for.
- **Rain has a real but small dampening effect.** Rainy days average ~17 fewer crimes than dry days (658 vs 676). Welch's t = -3.51, p ≈ 0.0005 — significant but tiny in absolute terms next to temperature.
- **Day-of-week matters more than weather.** Saturday/Friday run ~80 crimes/day above Sunday/Tuesday. This is why the day-of-week mean baseline is a serious benchmark for Phase 5, not a strawman.
- **The COVID dip is sharp.** Total crime drops noticeably after March 21, 2020 and stays low through that summer. I flag 2020 with `is_covid_year` instead of dropping it — methodologically more defensible than carving out a year.